In [13]:
import pandas as pd
from sklearn.ensemble  import RandomForestClassifier

# Modeling EDA

In [14]:
# Load file
df = pd.read_csv(r'C:\Users\peter\Documents\SJSU\Data Mining\CMPE255_TrafficCrashSeverityIndicator\data\processed\crashes+.csv')
df.head()

C:\Users\peter\AppData\Local\Temp\ipykernel_44540\859052757.py:2: DtypeWarning: Columns (0: SpeedingFlag, 1: HitAndRunFlag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\peter\Documents\SJSU\Data Mining\CMPE255_TrafficCrashSeverityIndicator\data\processed\crashes+.csv')


,CrashFactId,Name,MinorInjuries,ModerateInjuries,SevereInjuries,FatalInjuries,TcrNumber,CityDamageFlag,ShortFormFlag,Distance,...,SpeedingFlag,HitAndRunFlag,crash_year,crash_month,crash_day,crash_hour,crash_dayofweek,crash_date,Sobriety,injury_severity
0,591079,CR-0000071607,0,0,0,0,18-073-0962,True,False,228.0,...,NaN,NaN,2018,3,14,23,2,2018-03-14,NaN,0
1,591080,CR-0000071780,0,0,0,0,18-060-0123,True,False,148.0,...,NaN,NaN,2018,3,1,7,3,2018-03-01,Had Not Been Drinking,0
2,591081,CR-0000060418,0,0,0,0,16-033-0204,False,False,1583.0,...,NaN,NaN,2016,2,2,9,1,2016-02-02,Had Not Been Drinking,0
3,591082,CR-0000060410,0,1,0,0,16-041-0882,False,False,295.0,...,NaN,NaN,2016,2,10,20,2,2016-02-10,Had Been Drinking - Under Influence,2
4,591083,CR-0000060514,2,0,0,0,16-063-0761,False,False,0.0,...,NaN,NaN,2016,3,3,19,3,2016-03-03,Had Not Been Drinking,1


### Get distribution of predicted column
We will be training to predict injury severity. We need to know the distribution of our training data.

In [15]:
df.groupby('injury_severity').size()


injury_severity
0    41579
1    20011
2     9658
3     2293
4      654
dtype: int64

# Model 1: Decision Tree, One-Hot Encoding, All values of injury serverity, class_weights='balanced' 
### Filter down to trainable columns

In [16]:
df_filtered = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
df_encoded = pd.get_dummies(df_filtered, columns=['PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'], drop_first=True)


In [17]:
len(df_encoded.columns)

48

### Model Training

In [18]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('injury_severity', axis=1)  
y = df_encoded['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = RandomForestClassifier(class_weight='balanced', random_state=255, max_depth=25, n_estimators=100)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                                              feature  importance
43                     Sobriety_Had Not Been Drinking    0.099449
44                            Sobriety_Not Applicable    0.085645
4            PedestrianAction_No Pedestrians Involved    0.073595
38                             CollisionType_Rear End    0.061735
41                   CollisionType_Vehicle/Pedestrian    0.056725
39                            CollisionType_Sideswipe    0.042170
42       Sobriety_Had Been Drinking - Under Influence    0.040108
0        PedestrianAction_Crossing - Not In Crosswalk    0.039492
24                                  Lighting_Daylight    0.038010
40                         CollisionType_Vehicle/Bike    0.035180
27                                     Weather_Cloudy    0.035173
35                           CollisionType_Hit Object    0.034514
36                                CollisionType_Other    0.033805
22                       Lighting_Dark - Street Light    0.032265
13        

### Accuracy

In [19]:
from sklearn.metrics import accuracy_score

y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.50


In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.55      0.66      8316
           1       0.45      0.56      0.50      4002
           2       0.26      0.24      0.25      1932
           3       0.07      0.18      0.10       458
           4       0.05      0.44      0.09       131

    accuracy                           0.50     14839
   macro avg       0.33      0.40      0.32     14839
weighted avg       0.61      0.50      0.54     14839



Considering over half of the train and test set are non-injury crashes (injury_severity=0) this model is doing no better than a random guess.

# Model 1.2: Decision Tree, Get rid of injury_severity = 0, One hot encoding, class_weights='balanced'

In [21]:
df_filtered = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
# Keep only rows where injury_severity is NOT equal to 0
df_filtered = df_filtered[df_filtered['injury_severity'] != 0]
df_filtered.groupby('injury_severity').size()

injury_severity
1    20011
2     9658
3     2293
4      654
dtype: int64

In [22]:
df_encoded = pd.get_dummies(df_filtered, columns=['PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'], drop_first=True)

In [23]:
X = df_encoded.drop('injury_severity', axis=1)  
y = df_encoded['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = RandomForestClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                                              feature  importance
43                     Sobriety_Had Not Been Drinking    0.097583
38                             CollisionType_Rear End    0.079631
4            PedestrianAction_No Pedestrians Involved    0.072965
41                   CollisionType_Vehicle/Pedestrian    0.065448
27                                     Weather_Cloudy    0.046368
24                                  Lighting_Daylight    0.046164
0        PedestrianAction_Crossing - Not In Crosswalk    0.045754
42       Sobriety_Had Been Drinking - Under Influence    0.043935
22                       Lighting_Dark - Street Light    0.041540
35                           CollisionType_Hit Object    0.039112
36                                CollisionType_Other    0.036408
13                                 RoadwaySurface_Wet    0.034736
40                         CollisionType_Vehicle/Bike    0.030722
30                                       Weather_Rain    0.025570
44        

In [24]:
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.52


# Model 1.3: Decision Tree, Get rid of injury_severity = 0, label encoding, class_weights='balanced'

In [25]:
from sklearn.preprocessing import LabelEncoder

In [26]:
df_filtered13 = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
encoded_df13 = df_filtered13.copy()
for col in encoded_df13.select_dtypes(include='object').columns:
    if col != 'injury_severity':  # don't encode target
        encoded_df13[col] = LabelEncoder().fit_transform(df[col])

C:\Users\peter\AppData\Local\Temp\ipykernel_44540\422130791.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in encoded_df13.select_dtypes(include='object').columns:


In [27]:
X = encoded_df13.drop('injury_severity', axis=1)  
y = encoded_df13['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = RandomForestClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

            feature  importance
5     CollisionType    0.312270
6          Sobriety    0.239255
0  PedestrianAction    0.165641
3          Lighting    0.113065
4           Weather    0.079069
2  RoadwayCondition    0.052617
1    RoadwaySurface    0.038084


In [28]:
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.50
